## Importing Libraries

In [51]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import nltk 
nltk.download('wordnet')
nltk.download('punkt_tab')
from nltk.stem import WordNetLemmatizer
from transformers import AutoTokenizer

[nltk_data] Downloading package wordnet to /home/kleeat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/kleeat/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Loading Dataset

In [82]:
df = pd.read_csv('./../data/processed/20newsgroup_preprocessed_own.csv', on_bad_lines='skip', delimiter=";")
print(df.shape)
df.head()

(18828, 3)


,target,text,text_cleaned
0,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismresources altatheismarchive...
1,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismintroduction altatheismarch...
2,alt.atheism,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,article charley wingate writes well john quite...
3,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: R...,kings become philosophers philosophers become ...
4,alt.atheism,From: strom@Watson.Ibm.Com (Rob Strom)\nSubjec...,article pidaorg pidaorg bob mcgwier writes how...


In [83]:
df = df.dropna(subset=['text_cleaned'])
print(df.shape)

(18792, 3)


## Lemmatization

In [89]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = nltk.word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmas)

df['text_cleaned'] = df['text_cleaned'].apply(lambda x: lemmatize_text(x))
df.head()

,target,text,text_cleaned
0,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismresources altatheismarchive...
1,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismintroduction altatheismarch...
2,alt.atheism,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,article charley wingate writes well john quite...
3,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: R...,king become philosopher philosopher become kin...
4,alt.atheism,From: strom@Watson.Ibm.Com (Rob Strom)\nSubjec...,article pidaorg pidaorg bob mcgwier writes how...


## Tokenizing the documents

In [90]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=50000, stop_words="english", ngram_range = (1,2))

# Transform text data into TF-IDF features
X_tfidf = vectorizer.fit_transform(df['text_cleaned'])

# Show shape of transformed data
print("TF-IDF Matrix Shape:", X_tfidf.shape)

TF-IDF Matrix Shape: (18792, 50000)


Preview the vectorized documents

In [91]:
print(X_tfidf.toarray()[:5])
print(vectorizer.get_feature_names_out()[:20])

[[0.        0.        0.0568477 ... 0.        0.        0.       ]
 [0.        0.        0.        ... 0.        0.        0.       ]
 [0.        0.        0.        ... 0.        0.        0.       ]
 [0.        0.        0.        ... 0.        0.        0.       ]
 [0.        0.        0.        ... 0.        0.        0.       ]]
['aamir' 'aamir qazi' 'aap' 'aarhus' 'aaron' 'aaron bryce' 'aaron lung'
 'aaron ray' 'ab' 'abandon' 'abandoned' 'abandoning' 'abate' 'abbott'
 'abbreviation' 'abc' 'abc coverage' 'abc news'
 'abcdefghijklmnopqrstuvwxyz'
 'abcdefghijklmnopqrstuvwxyz abcdefghijklmnopqrstuvwxyz']


## Prepare dataset for training

In [92]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['target'], test_size=0.2, random_state=42)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

Train size: (15033, 50000) Test size: (3759, 50000)


## Training the model

In [93]:
# Initialize and train the model
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train, y_train)

MultinomialNB()

## Evaluate the model

In [94]:
# Predict on test set
y_pred = nb_classifier.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
print("Model Accuracy:", accuracy)
print("Model Precision:", precision)
print("Model Recall:", recall)
print("Model F1 Score:", f1)

Model Accuracy: 0.8494280393721735
Model Precision: 0.8654664698519513
Model Recall: 0.8351547076521697
Model F1 Score: 0.8345670059258083


In [61]:
new_texts = ["This means that any University desktop or laptop PCs which are still using Windows 10 will need to be upgraded to Windows 11 or replaced with more up-to-date equipment if they are unable to support Windows 11, as otherwise they will leave the University exposed to an increased cyber risk. This work is mandatory; any PCs which are still running Windows 10 will be blocked from the University network at some point in the future. Soon, colleagues and postgraduate researchers with Windows 10 PCs will be able to arrange their own upgrade, and choose a suitable time slot for this.", 
             "This week, a busy launch manifest saw SpaceX launch the Crew-10 and two Starlink missions from Florida and the delayed SPHEREx/PUNCH and Transporter 13 missions from California.Outside of the United States, Rocket Lab flew an Electron from New Zealand. A Chang Zhang 8 rocket launched from a new commercial pad in China which also launched two other rockets. A Russian Angara rocket launched an unknown payload from Plesetsk."]
lemmatized_new_texts = []
for text in new_texts:
    lemmatized_new_texts.append(lemmatize_text(text)) 
print(lemmatized_new_texts)
new_texts_tfidf = vectorizer.transform(lemmatized_new_texts)  # Transform using the trained vectorizer

# Assuming you have a dictionary mapping numbers to topics
target_names = df['target'].unique()  # Get category names
print(target_names)

['This mean that any University desktop or laptop PCs which are still using Windows 10 will need to be upgraded to Windows 11 or replaced with more up-to-date equipment if they are unable to support Windows 11 , a otherwise they will leave the University exposed to an increased cyber risk . This work is mandatory ; any PCs which are still running Windows 10 will be blocked from the University network at some point in the future . Soon , colleague and postgraduate researcher with Windows 10 PCs will be able to arrange their own upgrade , and choose a suitable time slot for this .', 'This week , a busy launch manifest saw SpaceX launch the Crew-10 and two Starlink mission from Florida and the delayed SPHEREx/PUNCH and Transporter 13 mission from California.Outside of the United States , Rocket Lab flew an Electron from New Zealand . A Chang Zhang 8 rocket launched from a new commercial pad in China which also launched two other rocket . A Russian Angara rocket launched an unknown payload

In [62]:
predictions = nb_classifier.predict(new_texts_tfidf)
print(predictions)

['comp.sys.mac.hardware' 'sci.space']


## BERT experiment

In [ ]:
# Load pre-trained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")